# Phun Xăm Vic — sổ tay phân tích

Logic pipeline đã chuyển hết sang package `pxv/` (có test, chạy được tự động).
Notebook này chỉ để **khám phá dữ liệu ad-hoc**, không còn là nơi chạy sản xuất.

Chạy pipeline đầy đủ:

```bash
python -m pxv.run_daily
```

Chạy test:

```bash
python -m pytest tests/ -v
```

> Các bước xử lý cũ (merge → MECE → funnel → tính thời gian) nằm ở
> `pxv/transform.py` và `pxv/marts.py`. Xem git history nếu cần bản notebook cũ.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pxv import io_local, marts, quality, transform

df_lead = io_local.load_leads()
df_hen  = io_local.load_appointments(df_lead)
df_inv  = io_local.load_invoices()

master = transform.build_master(df_lead, df_hen, df_inv)
print(f"{len(master):,} dòng | doanh thu {master['Doanh Thu (VNĐ)'].sum():,.0f} đ")

In [ ]:
# Chất lượng dữ liệu — xem trước khi tin bất kỳ con số nào
quality.run_checks(df_lead, df_inv, master).to_frame()

In [ ]:
# CLV theo kênh
dim = marts.build_dim_khach(master, df_inv)
(dim.groupby("Kênh Tiếp Cận")
    .agg(khách=("SĐT", "nunique"),
         doanh_thu=("tổng_doanh_thu", "sum"),
         CLV=("tổng_doanh_thu", "mean"))
    .sort_values("CLV", ascending=False)
    .round(0))

In [ ]:
# Funnel dịch vụ mồi
fm = marts.build_funnel_moi(df_inv)
print(f"{len(fm):,} khách mua dịch vụ mồi")
print(f"upsell trong 90 ngày: {fm['upsell_90d'].mean()*100:.1f}%")
fm.head()